# 🎙️ Fish Audio Drama 3 (35B-A3B) + Fish Speech WebUI в Google Colab (G4 96GB GPU Edition)

Оптимизированный ноутбук «всё в одном» для запуска флагманской модели синтеза речи, клонирования голоса и многоголосой озвучки сцен **[Fish Audio Drama 3 (35B-A3B)](https://huggingface.co/fishaudio/drama-3)** с официальным веб-интерфейсом **[Fish Speech Awesome WebUI](https://github.com/fishaudio/fish-speech)**, специально настроенный под видеокарту **G4 (96 ГБ VRAM)** в Google Colab.

### 💡 Почему видеокарта G4 (96 ГБ VRAM) идеальна для Fish Audio Drama 3
- **🔥 Аппаратное ускорение Block-FP8 (W8A8 Tensor Cores)**:
  - В отличие от старых карт SM80, современные GPU поколения Ada / Hopper / Blackwell (включая **G4 96GB**) поддерживают аппаратные инструкции FP8 (`float8_e4m3fn`). Все 256 экспертов MoE (35 млрд параметров) выполняются напрямую через высокоскоростное Triton-ядро `fused_moe_kernel`!
- **⚡ Полный стек CUDA Graphs (`1024 → 10240` токенов)**:
  - Объём в **96 ГБ видеопамяти** позволяет не экономить на буферах и включить **CUDA Graph декодирование (`FISH_DRAMA3_CUDA_GRAPH=1`)** сразу для всех бакетов контекста (до 10 240 фреймов). Это убирает задержки запуска ядер с CPU и даёт максимальную скорость генерации длинных диалогов.
- **🎛️ Выбор режима точности под 96 ГБ VRAM (`FP8 Turbo` или `BF16 Ultra`)**:
  - **Нативный FP8 (~42 ГБ из 96 ГБ VRAM с графами)** — максимальная скорость генерации.
  - **Полный BF16 (~76 ГБ из 96 ГБ VRAM)** — благодаря 96 ГБ памяти вы можете даже полностью распаковать все 35B параметров в `bfloat16`, что невозможно на 40 ГБ или 48 ГБ видеокартах!
- **🖥️ Официальный интерфейс Fish Speech Awesome WebUI**:
  - Автоматическая сборка современного TypeScript-интерфейса (`awesome_webui`) и привязка его к корневому адресу сервера (`/` и `/ui`).
- **💾 Полная интеграция с Google Drive**:
  - Веса модели (`~38 ГБ`) кэшируются в `MyDrive/FishSpeech_Models/drama-3` (скачиваются только при первом запуске).
  - Ваши референсные голоса сохраняются в `MyDrive/FishSpeech_References`, а сгенерированные записи — в `MyDrive/FishSpeech_Outputs`.
- **🚀 3 быстрых способа подключения к WebUI (без тормозов Cloudflare)**:
  - **Google Colab Direct Proxy**, **Pinggy Tunnel** и **Cloudflare Tunnel (HTTP/2)** выдаются одновременно.

> ⚠️ **Внимание**: Перед запуском убедитесь, что в верхнем меню **«Среда выполнения» (Runtime) → «Сменить среду выполнения» (Change runtime type)** выбран ускоритель **G4 GPU (96 GB)**.

## Step 1: Проверка окружения (G4 96GB GPU) и подключение Google Drive

In [ ]:
# Проверяем выделенный GPU, архитектуру CUDA и объём видеопамяти (ожидается ~96 ГБ)
!nvidia-smi
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"🔥 GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.1f} ГБ | Compute Capability: SM{props.major}{props.minor}")

# Сохранение кэша модели, референсных голосов и результатов на Google Drive
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive успешно подключён!")

## Step 2: Установка Fish Speech, сборка Awesome WebUI и оптимизация под G4 96GB

- Устанавливает аудио-библиотеки (`portaudio`, `sox`, `ffmpeg`) и зависимости `fishaudio/fish-speech`.
- Собирает официальный фронтенд **Awesome WebUI**.
- Настраивает поддержку аппаратного **FP8 (W8A8)** и безопасного фоллбэка **W8A16** для всех поколений GPU.

In [ ]:
import os
from pathlib import Path

print("📦 Установка системных аудио-библиотек (portaudio, sox, ffmpeg) и Node.js 22 LTS...")
!apt-get update -qq && apt-get install -y -qq portaudio19-dev libsox-dev ffmpeg xz-utils

# Устанавливаем свежий Node.js v22.14.0 (требуется Node 22.12+ для Vite и @tailwindcss/oxide)
!wget -q https://nodejs.org/dist/v22.14.0/node-v22.14.0-linux-x64.tar.xz -O /tmp/node22.tar.xz
!tar -xJf /tmp/node22.tar.xz -C /usr/local --strip-components=1
!rm -f /tmp/node22.tar.xz
!node -v && npm -v

%cd /content
if not os.path.exists("/content/fish-speech"):
    !git clone https://github.com/fishaudio/fish-speech.git
else:
    %cd /content/fish-speech
    !git pull

%cd /content/fish-speech
# Убираем из pyproject.toml жёсткий override protobuf<6.0.0, который ломает TensorBoard в Colab
pyproject_path = Path("/content/fish-speech/pyproject.toml")
if pyproject_path.exists():
    pyproject_path.write_text(
        pyproject_path.read_text(encoding="utf-8").replace("protobuf>=3.20.0,<6.0.0", "protobuf>=6.31.1"),
        encoding="utf-8"
    )

print("⚡ Быстрая установка Python-зависимостей через uv (без конфликта версий protobuf/torch)...")
!pip install -q uv
!uv pip install --system --no-config -q "huggingface_hub" "transformers>=5.0.0" "tokenizers>=0.21.0" \
    "hydra-core>=1.3.2" "natsort>=8.4.0" "einops>=0.7.0" "librosa>=0.10.1" \
    "kui>=1.6.0" "uvicorn>=0.30.0" "loguru>=0.6.0" "pyrootutils>=1.0.4" \
    "resampy>=0.4.3" "einx==0.2.2" pydub pyaudio "modelscope>=1.17.1" \
    "opencc-python-reimplemented==0.1.7" silero-vad ormsgpack "tiktoken>=0.8.0" \
    "pydantic>=2.9.2" cachetools safetensors "soundfile>=0.12.1" \
    "protobuf>=6.31.1" loralib julius argbind randomname markdown2 \
    flatten-dict pyloudnorm ffmpy pystoi torch-stoi importlib-resources
!uv pip install --system --no-config -q --no-deps descript-audio-codec descript-audiotools -e .
!uv pip install --system --no-config -q --no-deps --upgrade "protobuf>=6.31.1"

# Отключаем ненужный импорт TensorBoard внутри descript-audiotools
import site
for sp in site.getsitepackages():
    dec_file = Path(sp) / "audiotools" / "ml" / "decorators.py"
    if dec_file.exists():
        dec_file.write_text(
            dec_file.read_text(encoding="utf-8").replace(
                "from torch.utils.tensorboard import SummaryWriter",
                "SummaryWriter = None"
            ),
            encoding="utf-8"
        )

# Сборка официального веб-интерфейса Awesome WebUI на Node.js 22
webui_dist = Path("/content/fish-speech/awesome_webui/dist/index.html")
if not webui_dist.exists():
    print("🛠️ Сборка интерфейса Fish Speech Awesome WebUI (Node 22)...")
    %cd /content/fish-speech/awesome_webui
    !rm -rf node_modules package-lock.json
    !npm install --no-audit --no-fund
    !npm run build
    %cd /content/fish-speech
print(f"✅ WebUI собран: {webui_dist.exists()}")

# Патч 1: Открытие WebUI сразу по корневому адресу '/' (а также '/ui')
views_path = Path("/content/fish-speech/tools/server/views.py")
views_code = views_path.read_text(encoding="utf-8")
if '@routes.http("/")' not in views_code:
    views_code = views_code.replace(
        '@routes.http("/ui")\nclass WebUI(HttpView):',
        '@routes.http("/")\n@routes.http("/ui")\nclass WebUI(HttpView):'
    )
    views_path.write_text(views_code, encoding="utf-8")
    print("✅ Корневой маршрут '/' привязан к Awesome WebUI.")

# Патч 2: Универсальная поддержка FP8 на G4 (SM89/SM120 W8A8 Tensor Cores + безопасный фоллбэк W8A16)
fp8_path = Path("/content/fish-speech/fish_speech/models/text2semantic/fish_omni/fp8_moe.py")
moe_path = Path("/content/fish-speech/fish_speech/models/text2semantic/fish_omni/moe.py")
if fp8_path.exists() and moe_path.exists():
    fp8_code = fp8_path.read_text(encoding="utf-8")
    fp8_code = fp8_code.replace(
        "torch.cuda.get_device_capability(device) >= (8, 9)",
        "torch.cuda.get_device_capability(device) >= (8, 0)"
    )
    fp8_path.write_text(fp8_code, encoding="utf-8")

    moe_code = moe_path.read_text(encoding="utf-8")
    if "_USE_W8A8_TRITON" not in moe_code:
        moe_code = moe_code.replace(
            "if self.fp8 and x.is_cuda:",
            "if self.fp8 and x.is_cuda and torch.cuda.get_device_capability(x.device) >= (8, 9) and os.environ.get('FISH_DRAMA3_W8A16_FALLBACK', '0') != '1':"
        )
        if "import os" not in moe_code:
            moe_code = "import os\n" + moe_code
        moe_path.write_text(moe_code, encoding="utf-8")
    print("🚀 Настроено аппаратное ускорение FP8 Tensor Cores под G4 96GB!")

print("✅ Установка и настройка Fish Speech завершены!")

## Step 3: Загрузка весов `fishaudio/drama-3` и привязка папок Google Drive

- Если `USE_GOOGLE_DRIVE = True`:
  - **Кэш весов Drama 3**: `/content/drive/MyDrive/FishSpeech_Models/drama-3` (скачивается только при первом запуске)
  - **Референсные голоса**: `/content/drive/MyDrive/FishSpeech_References` (все загруженные в WebUI голоса сохраняются навсегда)
  - **Папка для готовых аудио**: `/content/drive/MyDrive/FishSpeech_Outputs`

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

# Опционально: укажите HF_TOKEN для максимальной скорости скачивания с Hugging Face
HF_TOKEN = ""  # @param {type:"string"}
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

os.makedirs("/content/fish-speech/checkpoints", exist_ok=True)

if USE_GOOGLE_DRIVE:
    DRIVE_MODELS_DIR = "/content/drive/MyDrive/FishSpeech_Models/drama-3"
    DRIVE_REFS_DIR = "/content/drive/MyDrive/FishSpeech_References"
    DRIVE_OUT_DIR = "/content/drive/MyDrive/FishSpeech_Outputs"
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
    os.makedirs(DRIVE_REFS_DIR, exist_ok=True)
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

    # Симлинк для чекпоинта drama-3
    local_ckpt = "/content/fish-speech/checkpoints/drama-3"
    if os.path.exists(local_ckpt) and not os.path.islink(local_ckpt):
        !rm -rf {local_ckpt}
    if not os.path.exists(local_ckpt):
        os.symlink(DRIVE_MODELS_DIR, local_ckpt)

    # Симлинк для референсных голосов
    local_refs = "/content/fish-speech/references"
    if os.path.exists(local_refs) and not os.path.islink(local_refs):
        !rm -rf {local_refs}
    if not os.path.exists(local_refs):
        os.symlink(DRIVE_REFS_DIR, local_refs)

    TARGET_CKPT_DIR = DRIVE_MODELS_DIR
    print(f"📁 Папки синхронизированы с Google Drive:\n  - Модель: {DRIVE_MODELS_DIR}\n  - Референсы: {DRIVE_REFS_DIR}\n  - Вывод: {DRIVE_OUT_DIR}")
else:
    TARGET_CKPT_DIR = "/content/fish-speech/checkpoints/drama-3"
    os.makedirs(TARGET_CKPT_DIR, exist_ok=True)
    os.makedirs("/content/fish-speech/references", exist_ok=True)

# Проверяем, все ли 15 шардов модели и codec.pth уже есть в кэше
ckpt_path = Path(TARGET_CKPT_DIR)
shards = list(ckpt_path.glob("model-*-of-00015.safetensors"))
has_codec = (ckpt_path / "codec.pth").exists()

if len(shards) == 15 and has_codec:
    print("⚡ Все 15 шардов модели fishaudio/drama-3 и codec.pth уже найдены на Google Drive! Скачивание пропускается.")
else:
    print("⬇️ Скачивание весов fishaudio/drama-3 с Hugging Face в Google Drive (~41 ГБ, только при первом запуске)...")
    snapshot_download(
        repo_id="fishaudio/drama-3",
        local_dir=TARGET_CKPT_DIR,
        cache_dir="/content/hf_temp_cache",
        local_dir_use_symlinks=False
    )
    # Удаляем временную папку .cache внутри Google Drive, чтобы не занимать лишнее место на Диске
    !rm -rf "{TARGET_CKPT_DIR}/.cache" /content/hf_temp_cache
    print("✅ Загрузка fishaudio/drama-3 на Google Drive успешно завершена!")

## Step 4: Запуск сервера Fish Speech (Drama 3 на G4 96GB) + Awesome WebUI и 3 быстрых туннеля

На видеокарте **G4 (96 ГБ VRAM)** по умолчанию включены:
- **`PRECISION_MODE = "FP8_TURBO"`** — аппаратный Block-FP8 (~42 ГБ VRAM) на максимальной скорости Tensor Cores (или выберите `"BF16_ULTRA"`, чтобы распаковать все 35B весов в `bfloat16` на ~76 ГБ из 96 ГБ VRAM!).
- **`ENABLE_CUDA_GRAPHS = True`** — захват всех бакетов CUDA Graph в 96 ГБ VRAM для мгновенного отклика при генерации длинной речи.

Сразу после прогрева модели появятся **3 быстрые ссылки для входа в WebUI**:
1. 🚀 **Google Colab Direct Proxy** (`*.googleusercontent.com/ui`) — максимальная скорость напрямую через серверы Google.
2. ⚡ **Pinggy Tunnel** (`https://xxxx.a.free.pinggy.link/ui`) — быстрый SSH/TCP-туннель без Cloudflare (отлично работает без VPN и с телефона).
3. 🌐 **Cloudflare Tunnel (HTTP/2)** (`https://xxxx.trycloudflare.com/ui`) — резервный туннель и эндпоинт для API.

In [ ]:
import os
import subprocess
import threading
import time
import re
import torch

# @title Настройки производительности G4 96GB
PRECISION_MODE = "FP8_TURBO (Аппаратный FP8 ~42 ГБ VRAM — Макс. скорость)"  # @param ["FP8_TURBO (Аппаратный FP8 ~42 ГБ VRAM — Макс. скорость)", "BF16_ULTRA (Полный BF16 ~76 ГБ из 96 ГБ VRAM)", "W8A16_SAFE (FP8 в памяти + BF16 вычисления)"]
ENABLE_CUDA_GRAPHS = True  # @param {type:"boolean"}

# Остановим старые процессы, если ячейку перезапускают
!pkill -f "tools/api_server.py" || true
!pkill -f "cloudflared" || true

# Убираем жёсткий override protobuf<6.0.0 из pyproject.toml и отключаем лишний импорт TensorBoard в audiotools
import site
from pathlib import Path
pyproject_path = Path("/content/fish-speech/pyproject.toml")
if pyproject_path.exists():
    pyproject_path.write_text(
        pyproject_path.read_text(encoding="utf-8").replace("protobuf>=3.20.0,<6.0.0", "protobuf>=6.31.1"),
        encoding="utf-8"
    )
!uv pip install --system --no-config -q flatten-dict pyloudnorm ffmpy pystoi torch-stoi importlib-resources
!uv pip install --system --no-config -q --no-deps --upgrade "protobuf>=6.31.1"
for sp in site.getsitepackages():
    dec_file = Path(sp) / "audiotools" / "ml" / "decorators.py"
    if dec_file.exists():
        dec_file.write_text(
            dec_file.read_text(encoding="utf-8").replace(
                "from torch.utils.tensorboard import SummaryWriter",
                "SummaryWriter = None"
            ),
            encoding="utf-8"
        )

# Скачивание Cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 96
env = os.environ.copy()

if PRECISION_MODE.startswith("BF16_ULTRA"):
    env["FISH_DRAMA3_FP8_EXPERTS"] = "0"
    env["FISH_DRAMA3_W8A16_FALLBACK"] = "0"
elif PRECISION_MODE.startswith("W8A16_SAFE"):
    env["FISH_DRAMA3_FP8_EXPERTS"] = "1"
    env["FISH_DRAMA3_W8A16_FALLBACK"] = "1"
else:
    env["FISH_DRAMA3_FP8_EXPERTS"] = "1"
    env["FISH_DRAMA3_W8A16_FALLBACK"] = "0"

env["FISH_DRAMA3_CUDA_GRAPH"] = "1" if ENABLE_CUDA_GRAPHS else "0"
env["PYTHONWARNINGS"] = "ignore::SyntaxWarning"

print(f"⚡ Запуск сервера Fish Speech (Drama 3) на G4 GPU ({vram_gb:.1f} ГБ VRAM)...")
print(f"   Режим: {PRECISION_MODE} | CUDA Graphs: {'Вкл' if ENABLE_CUDA_GRAPHS else 'Выкл'}")

%cd /content/fish-speech
api_proc = subprocess.Popen([
    "python", "tools/api_server.py",
    "--listen", "127.0.0.1:8080",
    "--llama-checkpoint-path", "checkpoints/drama-3",
    "--device", "cuda:0"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)

server_ready = threading.Event()

def log_server():
    for line in iter(api_proc.stdout.readline, ''):
        print("[FishSpeech]", line, end="")
        if "Startup done, listening server" in line or "Uvicorn running on" in line:
            server_ready.set()

threading.Thread(target=log_server, daemon=True).start()

print("⏳ Загрузка весов Drama 3 в 96 ГБ VRAM и захват CUDA Graphs (обычно 1–3 минуты)...")
server_ready.wait(timeout=300)

# 1. Встроенный прокси Google Colab (максимальная скорость в том же браузере)
try:
    from google.colab.output import eval_js
    colab_url = eval_js("google.colab.kernel.proxyPort(8080)")
except Exception:
    colab_url = None

# 2. Запуск Pinggy.io SSH-туннеля (быстрый аналог без Cloudflare)
pinggy_proc = subprocess.Popen([
    "ssh", "-p", "443",
    "-R0:127.0.0.1:8080",
    "-o", "StrictHostKeyChecking=no",
    "-o", "ServerAliveInterval=30",
    "a.pinggy.io"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

pinggy_url = None
def watch_pinggy():
    global pinggy_url
    for line in iter(pinggy_proc.stdout.readline, ''):
        m = re.search(r'https://[a-zA-Z0-9-]+\.a\.free\.pinggy\.link', line)
        if m and not pinggy_url:
            pinggy_url = m.group(0)
            print(f"\n⚡ [Pinggy Tunnel] Быстрая ссылка WebUI: {pinggy_url}/ui")

threading.Thread(target=watch_pinggy, daemon=True).start()

# 3. Запуск Cloudflare Tunnel в режиме HTTP/2
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8080",
    "--protocol", "http2"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

cf_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match and not cf_url:
        cf_url = match.group(0)
        time.sleep(2)
        print("\n" + "="*72)
        print("🎉 FISH SPEECH + DRAMA 3 (G4 96GB) ГОТОВ! ВЫБЕРИТЕ ССЫЛКУ ДЛЯ WEBUI:")
        if colab_url:
            base_colab = colab_url.rstrip('/')
            print(f"🚀 1. Google Colab Proxy (Макс. скорость): {base_colab}/ui")
        if pinggy_url:
            print(f"⚡ 2. Pinggy Tunnel (Без Cloudflare):      {pinggy_url}/ui")
        print(f"🌐 3. Cloudflare Tunnel (HTTP/2 + API):    {cf_url}/ui")
        print("="*72 + "\n")
        print("💡 Для вызовов по API (из client.py или агента) используйте базовый URL без /ui.")
        break

def drain_cf():
    for _ in iter(tunnel_proc.stdout.readline, ''):
        pass

threading.Thread(target=drain_cf, daemon=True).start()

try:
    api_proc.wait()
except KeyboardInterrupt:
    print("\n⏹️ Сервер остановлен.")
    api_proc.terminate()
    tunnel_proc.terminate()
    pinggy_proc.terminate()

## 🛑 (После завершения работы) Как гарантированно остановить списание CU

Когда закончите озвучку и тесты, **чтобы со 100% надёжностью остановить расход Compute Units (CU)**:
1. Нажмите **«Среда выполнения» (Runtime)** в верхнем меню Colab.
2. Выберите **«Отключиться и удалить среду выполнения» (Disconnect and delete runtime)**.